这个公式是深度学习（特别是卷积神经网络 CNN）中最基础、最重要的公式之一，用于计算**卷积层输出特征图（Feature Map）的空间尺寸**。

### 1. 公式含义拆解

$$ \text{output\_size} = \left\lfloor \frac{\text{input\_size} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1 $$

| 参数 | 含义 | 作用 |
| :--- | :--- | :--- |
| `input_size` | 输入特征图的宽或高 | 原始数据的空间维度 |
| `padding` | 填充大小 | 在输入边缘补零的圈数，用于控制输出尺寸和保留边缘信息 |
| `kernel_size` | 卷积核大小 | 滑动窗口的尺寸（如 3×3, 5×5） |
| `stride` | 步长 | 卷积核每次滑动的像素间距 |
| `+1` | — | 因为从位置0开始计数，所以需要加1 |
| `⌊ ⌋` | 向下取整 | 当分子不能被 stride 整除时，丢弃最后不完整的窗口 |

> **注意**：你给出的公式中没有显式写出向下取整符号 `⌊⌋`，但在实际实现（PyTorch / TensorFlow）中，结果一定是**向下取整**的整数。

### 2. 直观理解

可以把卷积想象成一个**滑动窗口**：

1.  **有效区域**：`input_size + 2*padding` 是填充后的总尺寸，减去 `kernel_size` 得到窗口可以"移动的范围"。
2.  **移动次数**：将移动范围除以 `stride`，得到窗口能完整滑动的次数。
3.  **起始位置**：加上初始位置（+1），就是输出的尺寸。

### 3. 常见特例

-   **Valid 卷积**（无填充）：`padding = 0`，输出会缩小
    $$\text{out} = \left\lfloor\frac{\text{in} - k}{s}\right\rfloor + 1$$
-   **Same 卷积**（保持尺寸不变）：令 `output_size = input_size`，反推所需 padding：
    $$\text{padding} = \left\lceil \frac{(k - 1) \times s}{2} \right\rceil$$
    当 `stride=1` 且 `k` 为奇数时，简化为 `padding = (k-1)/2`（如 3×3 → pad=1, 5×5 → pad=2）
-   **Stride > 1**：输出尺寸按比例缩小，常用于下采样替代池化层

### 4. 快速验证示例

| input | kernel | padding | stride | output |
| :--- | :--- | :--- | :--- | :--- |
| 32 | 3 | 1 | 1 | (32+2-3)/1+1 = **32** ✅ Same |
| 32 | 3 | 0 | 1 | (32+0-3)/1+1 = **30** ✅ Valid |
| 32 | 3 | 1 | 2 | (32+2-3)/2+1 = **16** ✅ 下采样 |
| 7 | 3 | 0 | 2 | (7+0-3)/2+1 = **3** ✅ ⌊4/2⌋+1 |

### 5. 补充说明

- 该公式对**宽和高分别独立计算**，非正方形输入/卷积核需分别代入。
- **通道数（channels）** 不由此公式决定，而是由卷积核的数量（`out_channels`）直接指定。
- PyTorch 中对应函数：`torch.nn.Conv2d`；TensorFlow 中：`tf.keras.layers.Conv2D`，它们内部都使用此公式自动推算输出形状。

掌握这个公式，就能在设计网络时精确控制每一层的特征图尺寸，避免维度不匹配的报错。